# 05 · Retrieval embeddings (Colab, GPU) — Day 5
Embeds train / val / test comments with `BAAI/bge-m3` in two views:
- `comment`          = text_enc
- `comment_caption`  = text_enc + "\n" + context_enc
Vectors are L2-normalised float16; retrieval on the laptop is exact cosine search (NumPy).
Writes to Drive: embeddings/bge-m3/{split}_{view}.npy, {split}_ids.txt, manifest.json.
Chunked and resumable: if Colab disconnects, re-run all cells.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q sentence-transformers==6.3.0 transformers==5.23.0

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
BASE    = Path("/content/drive/MyDrive/sinhala-sarcasm-llm")
IN_DIR  = BASE / "processed"
OUT_DIR = BASE / "embeddings" / "bge-m3"
CHUNK_DIR = OUT_DIR / "_chunks"
CHUNK_DIR.mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in IN_DIR.iterdir()))

Mounted at /content/drive
['test.jsonl', 'train.jsonl', 'val.jsonl']


In [ ]:
import json
import numpy as np
import pandas as pd

EXPECTED = {"train": 32_582, "val": 6_938, "test": 6_875}
data = {}
for split, n in EXPECTED.items():
    df = pd.read_json(IN_DIR / f"{split}.jsonl", lines=True, dtype=False)
    df["id"] = df["id"].astype(str)
    assert len(df) == n, f"{split}: {len(df)} rows, expected {n}"
    assert {"text_enc", "context_enc"} <= set(df.columns), "upload the notebook-01b versions of the files"
    df["v_comment"] = df["text_enc"].fillna("")
    ctx = df["context_enc"].fillna("").str.strip()
    df["v_comment_caption"] = np.where(ctx == "", df["v_comment"], df["v_comment"] + "\n" + ctx)
    data[split] = df
    print(f"{split:5s} {len(df):,} rows · mean chars: comment {df.v_comment.str.len().mean():.0f}, "
          f"comment+caption {df.v_comment_caption.str.len().mean():.0f}")

VIEWS = {"comment": "v_comment", "comment_caption": "v_comment_caption"}

train 32,582 rows · mean chars: comment 74, comment+caption 334
val   6,938 rows · mean chars: comment 73, comment+caption 343
test  6,875 rows · mean chars: comment 75, comment+caption 323


In [ ]:
import torch, sentence_transformers, transformers
from sentence_transformers import SentenceTransformer
from huggingface_hub import model_info

MODEL_ID = "BAAI/bge-m3"
MODEL_SHA = model_info(MODEL_ID).sha          # exact model revision, for reproducibility
model = SentenceTransformer(MODEL_ID, device="cuda", revision=MODEL_SHA)
model.half()
model.max_seq_length = 384                    # inputs are <= 256 XLM-R tokens (bge-m3 uses the same tokenizer)
print("model revision :", MODEL_SHA)
print("dimension      :", model.get_sentence_embedding_dimension())
print("GPU            :", torch.cuda.get_device_name(0))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (9 outputs left out)

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

model revision : 5617a9f61b028005a4858fdac845db406aefb181
dimension      : 1024
GPU            : Tesla T4


/tmp/ipykernel_2991/4126842371.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimension      :", model.get_sentence_embedding_dimension())


In [ ]:
CHUNK = 4096
BATCH = 64

def embed(split, view):
    out_path = OUT_DIR / f"{split}_{view}.npy"
    if out_path.exists():
        print(f"{out_path.name} exists — skipped"); return
    texts = data[split][VIEWS[view]].tolist()
    parts = []
    for start in range(0, len(texts), CHUNK):
        cpath = CHUNK_DIR / f"{split}_{view}_{start:06d}.npy"
        if not cpath.exists():
            vec = model.encode(texts[start:start + CHUNK], batch_size=BATCH, normalize_embeddings=True,
                               convert_to_numpy=True, show_progress_bar=False)
            np.save(cpath, vec.astype(np.float16))
        parts.append(np.load(cpath))
        print(f"  {split}/{view}: {min(start + CHUNK, len(texts)):,}/{len(texts):,}")
    np.save(out_path, np.concatenate(parts))
    print(f"wrote {out_path.name}")

In [ ]:
import time
t0 = time.time()
for split in ("train", "val", "test"):
    (OUT_DIR / f"{split}_ids.txt").write_text("\n".join(data[split].id) + "\n", encoding="utf-8")
    for view in VIEWS:
        embed(split, view)
print(f"done in {(time.time() - t0) / 60:.1f} min")

  train/comment: 4,096/32,582
  train/comment: 8,192/32,582
  train/comment: 12,288/32,582
  train/comment: 16,384/32,582
  train/comment: 20,480/32,582
  train/comment: 24,576/32,582
  train/comment: 28,672/32,582
  train/comment: 32,582/32,582
wrote train_comment.npy
  train/comment_caption: 4,096/32,582
  train/comment_caption: 8,192/32,582
  train/comment_caption: 12,288/32,582
  train/comment_caption: 16,384/32,582
  train/comment_caption: 20,480/32,582
  train/comment_caption: 24,576/32,582
  train/comment_caption: 28,672/32,582
  train/comment_caption: 32,582/32,582
wrote train_comment_caption.npy
  val/comment: 4,096/6,938
  val/comment: 6,938/6,938
wrote val_comment.npy
  val/comment_caption: 4,096/6,938
  val/comment_caption: 6,938/6,938
wrote val_comment_caption.npy
  test/comment: 4,096/6,875
  test/comment: 6,875/6,875
wrote test_comment.npy
  test/comment_caption: 4,096/6,875
  test/comment_caption: 6,875/6,875
wrote test_comment_caption.npy
done in 5.2 min


In [ ]:
checks = []
for split in ("train", "val", "test"):
    ids = (OUT_DIR / f"{split}_ids.txt").read_text(encoding="utf-8").split()
    assert ids == data[split].id.tolist(), f"{split}: id order mismatch"
    for view in VIEWS:
        E = np.load(OUT_DIR / f"{split}_{view}.npy").astype(np.float32)
        norms = np.linalg.norm(E, axis=1)
        checks.append({"split": split, "view": view, "rows": E.shape[0], "dim": E.shape[1],
                       "norm_min": norms.min(), "norm_max": norms.max(), "nan": int(np.isnan(E).sum())})
        assert E.shape[0] == len(ids) and not np.isnan(E).any()
display(pd.DataFrame(checks).round(4))

# Spot check: a training comment's nearest neighbour should be itself
E = np.load(OUT_DIR / "train_comment.npy").astype(np.float32)
q = E[:200]
assert (np.argmax(q @ E.T, axis=1) == np.arange(200)).mean() > 0.95
print("self-retrieval check: PASS")

manifest = {
    "model": MODEL_ID, "revision": MODEL_SHA, "dim": int(E.shape[1]), "dtype": "float16",
    "normalized": True, "max_seq_length": model.max_seq_length,
    "views": {"comment": "text_enc", "comment_caption": "text_enc + '\\n' + context_enc (text_enc only if no caption)"},
    "splits": {s: len(d) for s, d in data.items()},
    "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
    "sentence_transformers": sentence_transformers.__version__, "transformers": transformers.__version__,
    "created": pd.Timestamp.now(tz="UTC").isoformat(),
}
(OUT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps(manifest, indent=2))

,split,view,rows,dim,norm_min,norm_max,nan
0,train,comment,32582,1024,0.9998,1.0005,0
1,train,comment_caption,32582,1024,0.9998,1.0005,0
2,val,comment,6938,1024,0.9998,1.0005,0
3,val,comment_caption,6938,1024,0.9998,1.0005,0
4,test,comment,6875,1024,0.9998,1.0005,0
5,test,comment_caption,6875,1024,0.9998,1.0005,0


self-retrieval check: PASS
{
  "model": "BAAI/bge-m3",
  "revision": "5617a9f61b028005a4858fdac845db406aefb181",
  "dim": 1024,
  "dtype": "float16",
  "normalized": true,
  "max_seq_length": 384,
  "views": {
    "comment": "text_enc",
    "comment_caption": "text_enc + '\\n' + context_enc (text_enc only if no caption)"
  },
  "splits": {
    "train": 32582,
    "val": 6938,
    "test": 6875
  },
  "gpu": "Tesla T4",
  "torch": "2.14.1+cu130",
  "sentence_transformers": "6.3.0",
  "transformers": "5.23.0",
  "created": "2026-11-21T18:42:06.574329+00:00"
}


In [ ]:
import shutil
shutil.rmtree(CHUNK_DIR, ignore_errors=True)
print(sorted(p.name for p in OUT_DIR.iterdir()))

['manifest.json', 'test_comment.npy', 'test_comment_caption.npy', 'test_ids.txt', 'train_comment.npy', 'train_comment_caption.npy', 'train_ids.txt', 'val_comment.npy', 'val_comment_caption.npy', 'val_ids.txt']
